In [8]:
import os
os.chdir("/Users/adnan/Desktop/transit-analytics")

import pandas as pd
import requests
import os

os.chdir("/Users/adnan/Desktop/transit-analytics")

url = "https://data.ny.gov/resource/wujg-7c2s.csv"
params = {
    "$select": "date_trunc_ymd(transit_timestamp) AS transit_date, borough, transit_mode, SUM(ridership) AS daily_ridership, SUM(transfers) AS daily_transfers",
    "$where": "transit_timestamp >= '2024-01-01' AND transit_timestamp <= '2024-12-31'",
    "$group": "date_trunc_ymd(transit_timestamp), borough, transit_mode",
    "$order": "transit_date ASC",
    "$limit": 50000
}

print("Fetching pre-aggregated daily data...")
response = requests.get(url, params=params)
df = pd.read_csv(pd.io.common.StringIO(response.text))
print(f"Shape: {df.shape}")
print(f"\nDate range: {df['transit_date'].min()} to {df['transit_date'].max()}")
print(f"\nSample:\n{df.head()}")

Fetching pre-aggregated daily data...
Shape: (2196, 5)

Date range: 2024-01-01T00:00:00.000 to 2024-12-31T00:00:00.000

Sample:
              transit_date    borough transit_mode  daily_ridership  \
0  2024-01-01T00:00:00.000      Bronx       subway          93279.0   
1  2024-01-01T00:00:00.000   Brooklyn       subway         358682.0   
2  2024-01-01T00:00:00.000  Manhattan       subway         992681.0   
3  2024-01-01T00:00:00.000  Manhattan         tram          11724.0   
4  2024-01-01T00:00:00.000     Queens       subway         214262.0   

   daily_transfers  
0           3422.0  
1          10505.0  
2          15215.0  
3           2707.0  
4          15395.0  


In [6]:
import os
os.chdir("/Users/adnan/Desktop/transit-analytics")

import snowflake.connector
import pandas as pd
from snowflake.connector.pandas_tools import write_pandas

df = pd.read_csv("data/raw/mta_ridership.csv")
print(f"Loaded df with shape: {df.shape}")

conn = snowflake.connector.connect(
    user="USERNAME",
    password="PASSWORD",
    account="ACCOUNT",
    warehouse="TRANSIT_WH",
    database="TRANSIT_DB",
    schema="RAW"
)

cursor = conn.cursor()

cursor.execute("""
CREATE OR REPLACE TABLE TRANSIT_DB.RAW.MTA_RIDERSHIP (
    transit_timestamp     VARCHAR,
    transit_mode          VARCHAR,
    station_complex_id    VARCHAR,
    station_complex       VARCHAR,
    borough               VARCHAR,
    payment_method        VARCHAR,
    fare_class_category   VARCHAR,
    ridership             FLOAT,
    transfers             FLOAT,
    latitude              FLOAT,
    longitude             FLOAT,
    georeference          VARCHAR
)
""")

df.columns = [col.upper() for col in df.columns]

success, nchunks, nrows, _ = write_pandas(conn, df, "MTA_RIDERSHIP")
print(f"Loaded {nrows} rows in {nchunks} chunks — success: {success}")

conn.close()

Loaded df with shape: (480000, 11)


/Users/adnan/opt/anaconda3/envs/transit-bi/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


Loaded 480000 rows in 1 chunks — success: True


In [7]:
import snowflake.connector
import pandas as pd
import os

os.chdir("/Users/adnan/Desktop/transit-analytics")

conn = snowflake.connector.connect(
    user="USERNAME",
    password="PASSWORD",
    account="ACCOUNT",
    warehouse="TRANSIT_WH",
    database="TRANSIT_DB",
    schema="DBT_DEV"
)

marts = {
    "data/clean/mart_station_performance.csv": "SELECT * FROM mart_station_performance",
    "data/clean/mart_borough_trends.csv": "SELECT * FROM mart_borough_trends",
    "data/clean/mart_transit_mode.csv": "SELECT * FROM mart_transit_mode"
}

for path, query in marts.items():
    df = pd.read_sql(query, conn)
    df.to_csv(path, index=False)
    print(f"Exported {len(df)} rows → {path}")

conn.close()

/var/folders/nb/8t174vjx6lg6v6rvzj0zdjm80000gp/T/ipykernel_8426/1933975490.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Exported 428 rows → data/clean/mart_station_performance.csv
Exported 55 rows → data/clean/mart_borough_trends.csv
Exported 3 rows → data/clean/mart_transit_mode.csv


In [3]:
import pandas as pd
import requests
import os

os.chdir("/Users/adnan/Desktop/transit-analytics")

url = "https://data.ny.gov/resource/wujg-7c2s.csv"
params = {
    "$where": "transit_timestamp >= '2024-01-01' AND transit_timestamp <= '2024-12-31'",
    "$limit": 500000,
    "$order": "transit_timestamp ASC",
    "$select": "transit_timestamp,transit_mode,station_complex_id,station_complex,borough,payment_method,fare_class_category,ridership,transfers,latitude,longitude"
}

# Pull spread across the year by sampling every other month
dfs = []
for month in ['2024-01-01', '2024-03-01', '2024-05-01', '2024-07-01', '2024-09-01', '2024-11-01']:
    p = params.copy()
    p["$where"] = f"transit_timestamp >= '{month}' AND transit_timestamp < '{month[:7]}-28'"
    p["$limit"] = 80000
    r = requests.get(url, params=p)
    df_month = pd.read_csv(pd.io.common.StringIO(r.text))
    dfs.append(df_month)
    print(f"Fetched {len(df_month)} rows for {month}")

df = pd.concat(dfs, ignore_index=True)
df.columns = [col.upper() for col in df.columns]
print(f"\nTotal shape: {df.shape}")
df.to_csv("data/raw/mta_ridership.csv", index=False)
print("Saved to data/raw/mta_ridership.csv")

Fetched 80000 rows for 2024-01-01
Fetched 80000 rows for 2024-03-01
Fetched 80000 rows for 2024-05-01
Fetched 80000 rows for 2024-07-01
Fetched 80000 rows for 2024-09-01
Fetched 80000 rows for 2024-11-01

Total shape: (480000, 11)
Saved to data/raw/mta_ridership.csv


In [2]:
import os
os.chdir("/Users/adnan/Desktop/transit-analytics")

readme = """# NYC MTA Transit Performance Analytics Pipeline

An end-to-end analytics engineering project analyzing NYC subway and transit ridership patterns using Python, Snowflake, dbt, and Tableau Public.

## Live Dashboard
[View on Tableau Public](https://public.tableau.com/app/profile/adnan.ahsan/viz/MTAPerformanceAnalysis2024/Story?publish=yes)

## Problem Statement
The MTA serves millions of riders daily across NYC's five boroughs. This project builds a production-style analytics pipeline to identify ridership patterns by station, borough, and transit mode — giving transit planners and analysts a data-driven view of system performance.

## Stack
| Layer            | Tool                          |
|------------------|-------------------------------|
| Ingestion        | Python (requests, pandas)     |
| Data Warehouse   | Snowflake                     |
| Transformation   | dbt (staging → intermediate → marts) |
| Visualization    | Tableau Public                |
| Version Control  | Git / GitHub                  |

## Pipeline Architecture
MTA API → Python → Snowflake (RAW) → dbt → Snowflake (DBT_DEV) → Tableau Public

## Project Structure
transit-analytics/
├── data/
│   ├── raw/          ← raw API pull (not tracked in git)
│   └── clean/        ← mart CSVs exported for Tableau
├── ingestion/        ← Python ingestion scripts
└── notebooks/        ← exploration and loading notebooks
transit_dbt/
├── models/
│   ├── staging/      ← stg_mta_ridership (clean + type cast)
│   ├── intermediate/ ← int_daily_ridership (hourly → daily)
│   └── marts/        ← station performance, borough trends, transit mode
├── tests/            ← dbt schema tests
└── dbt_project.yml

## dbt Models
| Model | Layer | Description |
|-------|-------|-------------|
| stg_mta_ridership | Staging | Cleans raw MTA data, casts types, filters nulls |
| int_daily_ridership | Intermediate | Aggregates hourly records to daily by station |
| mart_station_performance | Mart | Total and avg daily ridership per station |
| mart_borough_trends | Mart | Daily ridership trends by borough |
| mart_transit_mode | Mart | Ridership breakdown by transit mode |

## dbt Tests
- 10/10 tests passing
- not_null, unique, and accepted_values tests across all mart models

## Key Findings
1. **Times Square is the busiest station by 2x** — handling over 700k riders vs ~350k for Grand Central in second place
2. **Subway accounts for the vast majority of MTA ridership** — tram and Staten Island Railway represent a small fraction
3. **Manhattan dominates borough ridership** — but Brooklyn shows the strongest weekday consistency
4. **Ridership dips are visible on weekends** across all boroughs, validating the data quality

## Data Source
- **Provider:** MTA via NYC Open Data (Socrata API)
- **Dataset:** MTA Subway Hourly Ridership — Beginning February 2022
- **Link:** https://data.ny.gov/Transportation/MTA-Subway-Hourly-Ridership-Beginning-February-2022/wujg-7c2s
- **Period:** 2024 full year (pre-aggregated daily totals)
"""

with open("README.md", "w") as f:
    f.write(readme)

print("README.md written.")


README.md written.


In [3]:
gitignore = """venv/
__pycache__/
*.pyc
.DS_Store
data/raw/
*.ipynb_checkpoints
.env
"""

with open(".gitignore", "w") as f:
    f.write(gitignore)

print(".gitignore written.")

.gitignore written.
